In [1]:
import itertools
from multiprocessing.pool import Pool

import pandas as pd
from sklearn import metrics
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.database import Model
from april.database import get_engine
from april.enums import Base
from april.enums import Heuristic
from april.enums import Strategy
from april.evaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
Creating Evaluation table


In [2]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [3]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    # print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    # print(f"{e} loaded.")

    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        _params.append([e, base, heuristic, strategy])

    return [_e for p in _params for _e in _evaluate(p)]

In [4]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(models)
evaluations = []
# with Pool() as p:
#     for e in tqdm(p.imap(evaluate, models), total=len(models), desc='Evaluate'):
#         evaluations.append(e)
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

['p2p-0.3-1_dae_20250417-125304.814194', 'p2p-0.3-1_daeltn_20250417-125313.792936', 'p2p-0.3-1_daeltnfrozen_20250417-125525.876858']


Evaluate:   0%|          | 0/3 [00:00<?, ?it/s]

Loading model p2p-0.3-1_dae_20250417-125304.814194 for event log p2p-0.3-1
Loading model p2p-0.3-1_daeltn_20250417-125313.792936 for event log p2p-0.3-1
Loading model p2p-0.3-1_daeltnfrozen_20250417-125525.876858 for event log p2p-0.3-1


## Pickle the results

In [5]:
out_dir = PLOT_DIR / 'isj-2019'
eval_file = out_dir / 'eval.pkl'

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
print(vars(evaluations[0]))
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState object at 0x0000013D3CF9F520>, 'recall': 0.9875595553202753, 'attribute_name': 'name', 'perspective': 'Control Flow', 'strategy': 'single', 'base': 'scores', 'file_name': 'p2p-0.3-1_dae_20250417-125304.814194.keras', 'id': 1, 'f1': 0.9831357048748353, 'precision': 0.9787513116474291, 'label': 'Normal', 'heuristic': 'best', 'axis': 0, 'model_id': 1}


  0%|          | 0/864 [00:00<?, ?it/s]

In [6]:
evaluation.to_pickle(eval_file)

In [7]:
display(evaluation)

,file_name,date,hyperparameters,training_duration,training_host,ad,dataset_name,process_model,noise,dataset_id,axis,base,heuristic,strategy,label,attribute_name,perspective,precision,recall,f1
0,p2p-0.3-1_dae_20250417-125304.814194.keras,2025-04-17 12:53:12.188518,"{'epochs': 10, 'batch_size': 100}",7.374324,Dev-RTX,DAE,p2p-0.3-1,p2p,0.3,1,0,scores,best,single,Normal,name,Control Flow,0.978751,0.987560,0.983136
1,p2p-0.3-1_dae_20250417-125304.814194.keras,2025-04-17 12:53:12.188518,"{'epochs': 10, 'batch_size': 100}",7.374324,Dev-RTX,DAE,p2p-0.3-1,p2p,0.3,1,0,scores,best,single,Anomaly,name,Control Flow,0.960438,0.933715,0.946888
2,p2p-0.3-1_dae_20250417-125304.814194.keras,2025-04-17 12:53:12.188518,"{'epochs': 10, 'batch_size': 100}",7.374324,Dev-RTX,DAE,p2p-0.3-1,p2p,0.3,1,0,scores,best,single,Normal,user,Data,0.904800,1.000000,0.950021
3,p2p-0.3-1_dae_20250417-125304.814194.keras,2025-04-17 12:53:12.188518,"{'epochs': 10, 'batch_size': 100}",7.374324,Dev-RTX,DAE,p2p-0.3-1,p2p,0.3,1,0,scores,best,single,Anomaly,user,Data,0.000000,0.000000,0.000000
4,p2p-0.3-1_dae_20250417-125304.814194.keras,2025-04-17 12:53:12.188518,"{'epochs': 10, 'batch_size': 100}",7.374324,Dev-RTX,DAE,p2p-0.3-1,p2p,0.3,1,1,scores,best,single,Normal,name,Control Flow,0.985739,0.958497,0.971927
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,p2p-0.3-1_daeltnfrozen_20250417-125525.876858....,2025-04-17 12:57:29.663168,"{'epochs': 10, 'batch_size': 100, 'epochs_ltn'...",123.786310,Dev-RTX,DAELTNFROZEN,p2p-0.3-1,p2p,0.3,1,1,scores,stable_right,position_attribute,Anomaly,user,Data,0.486111,0.041766,0.076923
860,p2p-0.3-1_daeltnfrozen_20250417-125525.876858....,2025-04-17 12:57:29.663168,"{'epochs': 10, 'batch_size': 100, 'epochs_ltn'...",123.786310,Dev-RTX,DAELTNFROZEN,p2p-0.3-1,p2p,0.3,1,2,scores,stable_right,position_attribute,Normal,name,Control Flow,0.956746,0.997735,0.976811
861,p2p-0.3-1_daeltnfrozen_20250417-125525.876858....,2025-04-17 12:57:29.663168,"{'epochs': 10, 'batch_size': 100, 'epochs_ltn'...",123.786310,Dev-RTX,DAELTNFROZEN,p2p-0.3-1,p2p,0.3,1,2,scores,stable_right,position_attribute,Anomaly,name,Control Flow,0.540000,0.055670,0.100935
862,p2p-0.3-1_daeltnfrozen_20250417-125525.876858....,2025-04-17 12:57:29.663168,"{'epochs': 10, 'batch_size': 100, 'epochs_ltn'...",123.786310,Dev-RTX,DAELTNFROZEN,p2p-0.3-1,p2p,0.3,1,2,scores,stable_right,position_attribute,Normal,user,Data,0.984884,0.999293,0.992036


In [8]:
print("Done")

Done
